# 05 — Evaluation & Interpretability

**Project:** Forecasting the Effects of Exchange Rate Fluctuations on US Trade Flows  
**Phase:** CRISP-DM 5 — Evaluation  
**Author:** Francisco Giordano Rigon

---

## Purpose

This notebook answers the central research question of the thesis:

> **How do exchange rate fluctuations affect US bilateral trade flows with Canada, Mexico, and Brazil?**

The 72 models in `04_modeling.ipynb` predict *how much* trade will occur.  
This notebook extracts *why* — specifically the role of exchange rate variables.

## Structure

1. Setup — load models and results
2. Feature Importance — which of the 73 variables matter most (RF and LightGBM)
3. SHAP Analysis — direction and magnitude of each variable's effect on predictions
4. Exchange Rate Focus — isolate FX and REER effects across countries and sectors
5. ARIMA vs ML gap — what the exchange rate features add beyond series history
6. Sector Sensitivity — which sectors respond most to exchange rate movements
7. Summary for thesis

---
## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import warnings
from pathlib import Path

import shap

warnings.filterwarnings('ignore')
np.random.seed(42)

# --- Paths ---
PROC    = Path('../data/processed')
MODELS  = Path('../models')
RESULTS = Path('../results')
FIGS    = RESULTS / 'figures' / 'evaluation'
FIGS.mkdir(parents=True, exist_ok=True)

# --- Config ---
PARTNERS = {'CAN': 'Canada', 'MEX': 'Mexico', 'BRA': 'Brazil'}
TARGETS  = [
    'exports_total', 'exports_commodities', 'exports_manufactured_goods', 'exports_high-tech',
    'imports_total', 'imports_commodities', 'imports_manufactured_goods', 'imports_high-tech',
]
TRAIN_END  = '2021-12-01'
TEST_START = '2022-01-01'

# Exchange rate and REER column prefixes (the thesis focus)
FX_COLS   = ['FX_USD_CAD', 'FX_USD_MXN', 'FX_USD_BRL']
REER_COLS = ['REER_USA', 'REER_CAN', 'REER_MEX', 'REER_BRA']

print('Setup complete.')

---
## 2. Load Data and Models

In [ ]:
# --- Load processed datasets ---
datasets = {}
for iso in PARTNERS:
    df = pd.read_csv(PROC / f'dataset_{iso.lower()}.csv', index_col=0, parse_dates=True)
    datasets[iso] = df

# --- Load all 48 ML models (RF + LightGBM) ---
rf_models   = {}
lgbm_models = {}

for iso in PARTNERS:
    for target in TARGETS:
        fname = MODELS / 'random_forest' / f'rf_{iso.lower()}_{target}.pkl'
        with open(fname, 'rb') as f:
            rf_models[(iso, target)] = pickle.load(f)['model']

        fname = MODELS / 'lightgbm' / f'lgbm_{iso.lower()}_{target}.pkl'
        with open(fname, 'rb') as f:
            lgbm_models[(iso, target)] = pickle.load(f)['model']

# --- Load combined metrics ---
df_all = pd.read_csv(RESULTS / 'forecasts' / 'metrics_all.csv')

print(f'Datasets: {list(datasets.keys())}')
print(f'RF models: {len(rf_models)}')
print(f'LightGBM models: {len(lgbm_models)}')
print(f'Metrics table: {df_all.shape}')

---
## 3. Feature Importance

Feature importance shows which of the 73 input variables most influenced each model's predictions.

This is the **first layer** of answering the research question:
- If FX and REER variables appear at the top → exchange rate is a strong predictor
- If lag variables dominate → recent history matters more than current macro conditions
- If crisis dummies appear → structural breaks (2008, COVID) shape trade flows heavily

We compute mean importance across all 24 RF models and all 24 LightGBM models.

In [ ]:
def get_feature_cols(df):
    exclude = [c for c in df.columns if 'exports' in c or 'imports' in c]
    return [c for c in df.columns if c not in exclude]

feature_cols = get_feature_cols(datasets['CAN'])

# --- Aggregate feature importance across all 24 RF models ---
rf_imp_all = pd.DataFrame(index=feature_cols)
for iso in PARTNERS:
    for target in TARGETS:
        rf_imp_all[f'{iso}_{target}'] = rf_models[(iso, target)].feature_importances_

rf_imp_mean = rf_imp_all.mean(axis=1).sort_values(ascending=False)

# --- Aggregate feature importance across all 24 LightGBM models ---
lgbm_imp_all = pd.DataFrame(index=feature_cols)
for iso in PARTNERS:
    for target in TARGETS:
        lgbm_imp_all[f'{iso}_{target}'] = lgbm_models[(iso, target)].feature_importances_

lgbm_imp_mean = lgbm_imp_all.mean(axis=1).sort_values(ascending=False)

print('=== Top 15 features — Random Forest (mean across 24 models) ===')
print(rf_imp_mean.head(15).to_string())
print()
print('=== Top 15 features — LightGBM (mean across 24 models) ===')
print(lgbm_imp_mean.head(15).to_string())

In [ ]:
# --- Plot Top 15 feature importances side by side ---
# Exchange rate and REER columns highlighted in red
fx_reer_cols = [c for c in feature_cols
                if any(fx in c for fx in FX_COLS + REER_COLS)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, imp, title in zip(axes,
                           [rf_imp_mean.head(15), lgbm_imp_mean.head(15)],
                           ['Random Forest', 'LightGBM']):
    colors = ['#C44E52' if c in fx_reer_cols else '#4C72B0' for c in imp.index]
    imp[::-1].plot(kind='barh', ax=ax, color=colors[::-1])
    ax.set_title(f'{title} — Top 15 Features\n(red = exchange rate / REER)', fontsize=12)
    ax.set_xlabel('Mean Feature Importance')
    ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(FIGS / 'feature_importance_top15.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/evaluation/feature_importance_top15.png')

---
## 4. SHAP Analysis

SHAP (SHapley Additive exPlanations) goes beyond feature importance:
- **Direction**: does a higher USD/BRL increase or decrease trade?
- **Magnitude**: by how much, on average?
- **Non-linearity**: does the effect change at different exchange rate levels?

This is the **core answer** to the thesis research question.

We compute SHAP values on the **test set (2022–2024)** for the Random Forest models.

> Install shap if needed: `pip install shap`

In [ ]:
# --- Helper: get X_test for one series ---
def get_X_test(df, target):
    feat_cols  = get_feature_cols(df)
    log_target = 'log_' + target
    clean = df[feat_cols + [log_target]].dropna()
    X = clean[feat_cols]
    return X[X.index >= TEST_START]

# --- Compute SHAP values for all 24 RF models ---
shap_rf_values = {}  # (iso, target) -> ndarray (36, 73)
shap_rf_X_test = {}  # (iso, target) -> DataFrame

for iso in PARTNERS:
    df = datasets[iso]
    for target in TARGETS:
        model  = rf_models[(iso, target)]
        X_test = get_X_test(df, target)
        explainer = shap.TreeExplainer(model)
        shap_vals = explainer.shap_values(X_test)
        shap_rf_values[(iso, target)] = shap_vals
        shap_rf_X_test[(iso, target)] = X_test
        print(f'SHAP RF: {iso} {target} -> shape={shap_vals.shape}')

print('\nAll SHAP values computed.')

In [ ]:
# --- SHAP summary plot: all 24 RF models aggregated ---
# Each dot = one test observation from one model
# Color = feature value (blue=low, red=high)
# X axis = SHAP value (positive = pushed prediction higher)

all_shap = np.vstack([shap_rf_values[(iso, t)] for iso in PARTNERS for t in TARGETS])
all_X    = pd.concat([shap_rf_X_test[(iso, t)] for iso in PARTNERS for t in TARGETS])

plt.figure(figsize=(10, 10))
shap.summary_plot(all_shap, all_X, max_display=20, show=False)
plt.title('SHAP Summary — Random Forest (all 24 models, test 2022-2024)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / 'shap_rf_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/evaluation/shap_rf_summary.png')

---
## 5. Exchange Rate Focus

Zoom in on FX and REER variables.

**SHAP dependence plot:** shows how the model's prediction changes as the exchange rate changes.  
- X axis = exchange rate value  
- Y axis = SHAP contribution (positive = predicted more trade; negative = less trade)  
- Expected sign: USD appreciation (FX rises) → exports fall (negative SHAP) → imports rise (positive SHAP)

In [ ]:
# --- SHAP dependence plots: bilateral FX rate for exports_total and imports_total ---
fx_var_map = {'CAN': 'FX_USD_CAD', 'MEX': 'FX_USD_MXN', 'BRA': 'FX_USD_BRL'}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for row_idx, target in enumerate(['exports_total', 'imports_total']):
    for col_idx, iso in enumerate(PARTNERS):
        ax = axes[row_idx, col_idx]
        fx_col = fx_var_map[iso]

        shap_vals = shap_rf_values[(iso, target)]
        X_test    = shap_rf_X_test[(iso, target)]

        feat_list = list(X_test.columns)
        if fx_col in feat_list:
            fx_idx   = feat_list.index(fx_col)
            fx_vals  = X_test[fx_col].values
            shap_fx  = shap_vals[:, fx_idx]

            color = '#C44E52' if 'exports' in target else '#4C72B0'
            ax.scatter(fx_vals, shap_fx, alpha=0.7, s=40, color=color)
            ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
            ax.set_xlabel(fx_col)
            ax.set_ylabel('SHAP value (log trade)')
            ax.set_title(f'{iso} — {target.replace("_", " ")}', fontsize=10)
            ax.grid(alpha=0.3)

fig.suptitle('Bilateral FX Rate vs SHAP Contribution (RF)\nPositive = more trade predicted; Negative = less trade',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGS / 'shap_fx_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/evaluation/shap_fx_dependence.png')

---
## 6. ARIMA vs ML Gap — What Do Exchange Rate Features Add?

ARIMA uses **only the series history** — no knowledge of exchange rates.  
RF and LightGBM use **73 features including exchange rates**.

The MAPE gap between ARIMA and best ML model per series is a proxy for:
> *How much predictive value do the exchange rate and macro features add beyond the series own history?*

Larger gap = exchange rate features explain more of the trade variation in that series.

In [ ]:
# --- MAPE gap: ARIMA vs best ML per series ---
df_arima_m = (df_all[df_all['algorithm'] == 'ARIMA']
              [['country', 'target', 'MAPE']]
              .rename(columns={'MAPE': 'MAPE_ARIMA'}))
df_ml_best = (df_all[df_all['algorithm'] != 'ARIMA']
              .groupby(['country', 'target'])['MAPE']
              .min().reset_index()
              .rename(columns={'MAPE': 'MAPE_best_ML'}))

df_gap = df_arima_m.merge(df_ml_best, on=['country', 'target'])
df_gap['gap_pp']  = (df_gap['MAPE_ARIMA'] - df_gap['MAPE_best_ML']).round(4)
df_gap['pct_improvement'] = (df_gap['gap_pp'] / df_gap['MAPE_ARIMA'] * 100).round(1)
df_gap = df_gap.sort_values('gap_pp', ascending=False)

print('=== MAPE Gap: ARIMA vs Best ML Model ===')
print('Positive gap = ML is better; larger = exchange rate features helped more')
print()
print(df_gap.to_string(index=False))
print()
print(f'Average gap: {df_gap["gap_pp"].mean():.4f} pp ({df_gap["pct_improvement"].mean():.1f}% reduction)')

In [ ]:
# --- Plot: gap by sector ---
sectors    = ['total', 'commodities', 'manufactured_goods', 'high-tech']
directions = ['exports', 'imports']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, direction in zip(axes, directions):
    vals = []
    for sector in sectors:
        target = f'{direction}_{sector}'
        vals.append(df_gap[df_gap['target'] == target]['gap_pp'].mean())

    bars = ax.bar(sectors, vals, color='#55A868', edgecolor='white')
    ax.set_title(f'{direction.capitalize()} — MAPE gap by sector', fontsize=11)
    ax.set_ylabel('MAPE reduction (pp)')
    ax.tick_params(axis='x', rotation=15)
    ax.grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)

fig.suptitle('Accuracy gain from adding exchange rate features (ARIMA → best ML)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGS / 'arima_ml_gap_by_sector.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/evaluation/arima_ml_gap_by_sector.png')

---
## 7. Sector Sensitivity to Exchange Rate

Which sectors are most sensitive to exchange rate movements?

Proxy: **mean absolute SHAP value** of FX and REER variables for each series.  
Higher value = exchange rate changes explain more of the prediction variance in that sector.

In [ ]:
# --- FX sensitivity score per series ---
sensitivity = []

for iso in PARTNERS:
    for target in TARGETS:
        shap_vals = shap_rf_values[(iso, target)]
        X_test    = shap_rf_X_test[(iso, target)]
        feat_list = list(X_test.columns)

        fx_idx = [i for i, c in enumerate(feat_list) if c in fx_reer_cols]
        mean_abs_shap_fx = np.abs(shap_vals[:, fx_idx]).mean()

        sensitivity.append({
            'country': iso,
            'target':  target,
            'fx_sensitivity': round(mean_abs_shap_fx, 6),
        })

df_sens = pd.DataFrame(sensitivity).sort_values('fx_sensitivity', ascending=False)

print('=== Exchange Rate Sensitivity by Series (mean |SHAP| of FX+REER variables) ===')
print()
print(df_sens.to_string(index=False))

In [ ]:
# --- Heatmap: FX sensitivity by country x sector ---
pivot = df_sens.pivot_table(values='fx_sensitivity', index='target', columns='country')

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Mean |SHAP| of FX+REER features')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([t.replace('_', ' ') for t in pivot.index])
ax.set_title('Exchange Rate Sensitivity by Sector and Country\n(Random Forest — test 2022-2024)', fontsize=12)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.4f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / 'fx_sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/figures/evaluation/fx_sensitivity_heatmap.png')

---
## 8. Summary for Thesis

In [ ]:
print('=== THESIS SUMMARY — KEY FINDINGS ===')
print()
print('1. OVERALL MODEL PERFORMANCE (MAPE, test 2022-2024):')
for algo, mape in df_all.groupby('algorithm')['MAPE'].mean().sort_values().items():
    print(f'   {algo:<15}: {mape:.2f}%')

print()
print('2. WINS PER SERIES (best MAPE):')
best = df_all.loc[df_all.groupby(['country','target'])['MAPE'].idxmin()]
for algo, n in best['algorithm'].value_counts().items():
    print(f'   {algo:<15}: {n}/24 series')

print()
print('3. EXCHANGE RATE FEATURES VALUE (ARIMA vs ML gap):')
print(f'   Mean MAPE reduction: {df_gap["gap_pp"].mean():.4f} pp ({df_gap["pct_improvement"].mean():.1f}%)')
print(f'   Series where ARIMA beats ML: {(df_gap["gap_pp"] < 0).sum()}/24')

print()
print('4. TOP 5 MOST FX-SENSITIVE SERIES:')
for _, r in df_sens.head(5).iterrows():
    print(f'   {r["country"]} {r["target"]:<35}: sensitivity={r["fx_sensitivity"]:.5f}')

print()
print('Figures saved to: results/figures/evaluation/')
print('Ready for thesis Results and Discussion sections.')